In [1]:
import os
import csv
import re
import pandas as pd

import matplotlib.pyplot as plt
import numpy as np
import wave
import contextlib

In [2]:
original_csv_path = "BBCSoundEffects.csv"
processed_csv_path = "bbc_clocks.csv"

In [ ]:
#copy only clock items to new csv
print(os.getcwd())
with open(original_csv_path, newline="") as f:
    reader = csv.DictReader(f)
    with open(processed_csv_path, "w", newline="") as out_f:
        writer = csv.DictWriter(out_f, fieldnames=reader.fieldnames)
        writer.writeheader()
        for row in reader:
            if "clock" in row["CDName"].lower():
                writer.writerow(row)
            

In [ ]:
#copy item into current folder for testing
code = "07016036"
src_path = "/scratch/local/ssd/hani/bbc_clocks/audio/"

dst_path = os.path.join(os.getcwd(),"TEST.wav")
os.system(f"cp {os.path.join(src_path, code + '.wav')} {dst_path}")

with contextlib.closing(wave.open(dst_path,'r')) as f:
    frames = f.getnframes()
    rate = f.getframerate()
    nchannels = f.getnchannels()
    duration = frames / float(rate)
    print(f"Duration: {duration} seconds")
    signal = f.readframes(frames)
    signal = np.frombuffer(signal, dtype=np.int16)
    signal = signal.reshape(-1, nchannels)[:, 0]
    time = np.linspace(0, duration, num=frames)
    plt.figure(1)
    plt.title('Waveform of ' + code)
    plt.plot(time, signal)
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.show()
    

In [ ]:
#add repetitions column

df = pd.read_csv(processed_csv_path)

def extract_repetitions(description):
    if pd.isna(description):
        return -1
    
    match = re.search(r'(\w+)\s+o\'clock', description, re.IGNORECASE)
    if match:
        word = match.group(1)
        
        if word.isdigit():
            return int(word)
        
        word_to_num = {
            'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4,
            'five': 5, 'six': 6, 'seven': 7, 'eight': 8, 'nine': 9,
            'ten': 10, 'eleven': 11, 'twelve': 12
        }
        return word_to_num.get(word.lower(), -1)
    
    return -1

df['repetitions'] = df['description'].apply(extract_repetitions)

df.to_csv(processed_csv_path, index=False)

In [ ]:
#count -1 entries - unused files
count = 0
total = 0
with open(processed_csv_path, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row["repetitions"] == "-1":
            count += 1
        total += 1
print(f"Number of entries with -1 repetitions: {count}")
print(f"Total number of entries: {total}")

In [ ]:
#add start time column for manual labelling

df = pd.read_csv(processed_csv_path)
df['start_time'] = 0.0
df.to_csv(processed_csv_path, index=False)
